# 207 — BaseAgent
Verifies the abstract base class mechanics: construction, `log_tool_event`,
`log_decision_event`, and `generate_ai_summary` — using a `DummyAgent` subclass
against a real Neo4j-backed trace.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import uuid
import logging
import pandas as pd

from src.config import get_neo4j_settings
from src.storage.neo4j_repository import Neo4jRepository
from src.storage.trace_repository import TraceRepository
from src.tracing.trace_service import TraceService
from src.tools.trace_tools import TraceTools
from src.agents.base import BaseAgent
from src.domain.models import (
    AgentResult,
    InvestigationRequest,
    InvestigationTrace,
    UserContext,
)

# Route agent.* logs to the notebook console so we can see the debug lines.
logging.basicConfig(level=logging.DEBUG, format="%(name)-30s %(levelname)-8s %(message)s")

settings = get_neo4j_settings()
neo4j    = Neo4jRepository(**vars(settings))
neo4j.test_connection()

repo    = TraceRepository(neo4j)
service = TraceService(repo)
tools   = TraceTools(repo)
print("Connected")

neo4j.pool                     DEBUG    [#0000]  _: <POOL> created, routing address IPv4Address(('ef39cb3b.databases.neo4j.io', 7687))
neo4j                          DEBUG    [#0000]  _: <WORKSPACE> resolve home database
neo4j.pool                     DEBUG    [#0000]  _: <POOL> attempting to update routing table from IPv4Address(('ef39cb3b.databases.neo4j.io', 7687))
neo4j.io                       DEBUG    [#0000]  _: <RESOLVE> in: ef39cb3b.databases.neo4j.io:7687
neo4j.io                       DEBUG    [#0000]  _: <RESOLVE> dns resolver out: 35.244.112.230:7687
neo4j.pool                     DEBUG    [#0000]  _: <POOL> _acquire router connection, database=None, address=ResolvedIPv4Address(('35.244.112.230', 7687))
neo4j.pool                     DEBUG    [#0000]  _: <POOL> trying to hand out new connection
neo4j.io                       DEBUG    [#0000]  C: <OPEN> 35.244.112.230:7687
neo4j.io                       DEBUG    [#C072]  C: <SECURE> ef39cb3b.databases.neo4j.io
neo4j.io     

Connected


## Define DummyAgent

`DummyAgent` is the minimal concrete subclass — it implements the required
`run()` method and delegates everything else to `BaseAgent`.
Real agents (`GraphAgent`, `RiskAgent`, `TraceAgent`) follow exactly this
pattern.

In [3]:
class DummyAgent(BaseAgent):
    """Minimal concrete agent used to verify BaseAgent mechanics."""

    def run(
        self,
        task: str,
        context: dict,
        trace: InvestigationTrace,
    ) -> AgentResult:
        company = context.get("company_name", "unknown")

        # 1. Log a routing decision before doing any work.
        self.log_decision_event(
            trace,
            decision=f"Starting task '{task}' for {company}",
            why="entry point of DummyAgent.run",
        )

        # 2. Simulate calling a tool and logging the result.
        self.log_tool_event(
            trace,
            tool_name="dummy_check",
            input_summary=f"company_name={company}",
            output_summary="risk_level=LOW (dummy result)",
            decision="no further steps required",
            why="DummyAgent always returns LOW risk",
            step_id="step_1",
            entity_refs=[{"label": "Company", "name": company}],
        )

        # 3. Optionally generate an AI narrative (None if no client).
        ai_text = self.generate_ai_summary(
            system_prompt="You are a risk analyst.",
            user_prompt=f"Summarise a LOW-risk finding for {company} in one sentence.",
        )

        summary = ai_text or f"{company}: LOW risk (AI unavailable — no summary generated)."

        return AgentResult(
            request_id=trace.request_id,
            entity_name=company,
            success=True,
            summary=summary,
            findings={"risk_level": "LOW"},
            trace=trace,
        )


print("DummyAgent defined")

DummyAgent defined


## Verify BaseAgent is abstract

Instantiating `BaseAgent` directly must raise `TypeError`.

In [4]:
try:
    BaseAgent("should-fail", service)  # type: ignore[abstract]
    print("ERROR — BaseAgent should not be instantiable")
except TypeError as exc:
    print(f"✓ BaseAgent is abstract: {exc}")

✓ BaseAgent is abstract: Can't instantiate abstract class BaseAgent with abstract method run


## Instantiate DummyAgent — without AI client

In [5]:
agent = DummyAgent(name="dummy-agent", trace_service=service)
print(f"agent.name       : {agent.name}")
print(f"agent._ai_client : {agent._ai_client}")
print(f"logger name      : {agent._log.name}")

agent.name       : dummy-agent
agent._ai_client : None
logger name      : agent.dummy-agent


## Start a trace, run the agent

In [6]:
request = InvestigationRequest(
    entity_name="TELEFONICA O2 HOLDINGS LIMITED",
    context=UserContext(
        user_id="analyst-001",
        session_id=str(uuid.uuid4()),
        metadata={"mode": "interactive"},
    ),
    request_id=str(uuid.uuid4()),
)

trace = service.start_trace(request, request.context)
print(f"trace_id: {trace.request_id}")

neo4j                          DEBUG    [#0000]  _: <WORKSPACE> routing towards fixed database: neo4j
neo4j                          DEBUG    [#0000]  _: <WORKSPACE> pinning database: 'neo4j'
neo4j.pool                     DEBUG    [#0000]  _: <POOL> acquire routing connection, access_mode='WRITE', database=AcquisitionDatabase(name='neo4j', guessed=False)
neo4j.pool                     DEBUG    [#0000]  _: <POOL> routing aged?, database=neo4j
neo4j.pool                     DEBUG    [#0000]  _: <ROUTING> purge check: last_updated_time=34700.39359025, ttl=10, perf_time=34710.485115875 => False
neo4j.pool                     DEBUG    [#0000]  _: <ROUTING> checking table freshness (readonly=False): table expired=True, has_server_for_mode=True, table routers=OrderedSet((IPv4Address(('p-ef39cb3b-a2ee-0003.production-orch-1243.neo4j.io', 7687)))) => False
neo4j.pool                     DEBUG    [#0000]  _: <POOL> attempting to update routing table from IPv4Address(('p-ef39cb3b-a2ee-0003.produ

trace_id: 4cb03697-cf3c-4699-99ba-ba9ca5100e6d


In [7]:
result = agent.run(
    task="risk_check",
    context={"company_name": "TELEFONICA O2 HOLDINGS LIMITED"},
    trace=trace,
)
print(f"success : {result.success}")
print(f"summary : {result.summary}")
print(f"findings: {result.findings}")
print(f"in-memory events: {len(trace.events)}")

agent.dummy-agent              DEBUG    decision="Starting task 'risk_check' for TELEFONICA O2 HOLDINGS LIMITED" why='entry point of DummyAgent.run'
neo4j                          DEBUG    [#0000]  _: <WORKSPACE> routing towards fixed database: neo4j
neo4j                          DEBUG    [#0000]  _: <WORKSPACE> pinning database: 'neo4j'
neo4j.pool                     DEBUG    [#0000]  _: <POOL> acquire routing connection, access_mode='WRITE', database=AcquisitionDatabase(name='neo4j', guessed=False)
neo4j.pool                     DEBUG    [#0000]  _: <POOL> routing aged?, database=neo4j
neo4j.pool                     DEBUG    [#0000]  _: <ROUTING> purge check: last_updated_time=34710.513358833, ttl=10, perf_time=34726.170385 => False
neo4j.pool                     DEBUG    [#0000]  _: <ROUTING> checking table freshness (readonly=False): table expired=True, has_server_for_mode=True, table routers=OrderedSet((IPv4Address(('p-ef39cb3b-a2ee-0003.production-orch-1243.neo4j.io', 7687)))) =

success : True
summary : TELEFONICA O2 HOLDINGS LIMITED: LOW risk (AI unavailable — no summary generated).
findings: {'risk_level': 'LOW'}
in-memory events: 2


## Finalize and retrieve

In [8]:
service.finalize_trace(trace, final_summary=result.summary)

r = tools.retrieve_trace(trace.request_id)
print(f"[{'✓' if r.success else '✗'}] {r.tool_name}  ({r.duration_ms} ms)")
print(f"    {r.summary}")
if r.data:
    print(f"\nfinal_summary: {r.data['final_summary']}")
    print("\nEvents:")
    events_df = pd.DataFrame(r.data["events"])
    display(events_df[["event_number", "event_type", "agent_name", "tool_name", "input_summary", "output_summary", "decision"]])

neo4j                          DEBUG    [#0000]  _: <WORKSPACE> routing towards fixed database: neo4j
neo4j                          DEBUG    [#0000]  _: <WORKSPACE> pinning database: 'neo4j'
neo4j.pool                     DEBUG    [#0000]  _: <POOL> acquire routing connection, access_mode='WRITE', database=AcquisitionDatabase(name='neo4j', guessed=False)
neo4j.pool                     DEBUG    [#0000]  _: <POOL> routing aged?, database=neo4j
neo4j.pool                     DEBUG    [#0000]  _: <ROUTING> purge check: last_updated_time=34726.275557917, ttl=10, perf_time=34744.394951375 => False
neo4j.pool                     DEBUG    [#0000]  _: <ROUTING> checking table freshness (readonly=False): table expired=True, has_server_for_mode=True, table routers=OrderedSet((IPv4Address(('p-ef39cb3b-a2ee-0003.production-orch-1243.neo4j.io', 7687)))) => False
neo4j.pool                     DEBUG    [#0000]  _: <POOL> attempting to update routing table from IPv4Address(('p-ef39cb3b-a2ee-0003.prod

[✓] retrieve_trace  (47.2 ms)
    Trace '4cb03697-cf3c-4699-99ba-ba9ca5100e6d' for 'TELEFONICA O2 HOLDINGS LIMITED': 2 event(s).

final_summary: TELEFONICA O2 HOLDINGS LIMITED: LOW risk (AI unavailable — no summary generated).

Events:


,event_number,event_type,agent_name,tool_name,input_summary,output_summary,decision
0,1,agent_reasoning,dummy-agent,,Starting task 'risk_check' for TELEFONICA O2 H...,,Starting task 'risk_check' for TELEFONICA O2 H...
1,2,tool_returned,dummy-agent,dummy_check,company_name=TELEFONICA O2 HOLDINGS LIMITED,risk_level=LOW (dummy result),no further steps required


## Verify generate_ai_summary — without AI client

`generate_ai_summary` must return `None` silently when no client is set.

In [9]:
result_no_ai = agent.generate_ai_summary(
    system_prompt="You are a risk analyst.",
    user_prompt="Summarise the risk for TELEFONICA O2 HOLDINGS LIMITED.",
)
assert result_no_ai is None, "Expected None when ai_client is not set"
print("✓ generate_ai_summary returns None when ai_client is None")

agent.dummy-agent              DEBUG    generate_ai_summary skipped — no ai_client configured


✓ generate_ai_summary returns None when ai_client is None


## Verify agent_name is stored on Neo4j TraceEvent nodes

The `agent_name` column should read `"dummy-agent"` for every event
written by this agent.

In [10]:
rows = neo4j.run_query(
    """
    MATCH (t:InvestigationTrace {trace_id: $trace_id})-[:HAS_EVENT]->(e:TraceEvent)
    RETURN e.event_number AS event_number, e.event_type AS event_type,
           e.agent_name  AS agent_name
    ORDER BY e.event_number
    """,
    {"trace_id": trace.request_id},
)
display(pd.DataFrame(rows))

neo4j                          DEBUG    [#0000]  _: <WORKSPACE> routing towards fixed database: neo4j
neo4j                          DEBUG    [#0000]  _: <WORKSPACE> pinning database: 'neo4j'
neo4j.pool                     DEBUG    [#0000]  _: <POOL> acquire routing connection, access_mode='WRITE', database=AcquisitionDatabase(name='neo4j', guessed=False)
neo4j.pool                     DEBUG    [#0000]  _: <POOL> routing aged?, database=neo4j
neo4j.pool                     DEBUG    [#0000]  _: <ROUTING> purge check: last_updated_time=34744.419204167, ttl=10, perf_time=34757.951266292 => False
neo4j.pool                     DEBUG    [#0000]  _: <ROUTING> checking table freshness (readonly=False): table expired=True, has_server_for_mode=True, table routers=OrderedSet((IPv4Address(('p-ef39cb3b-a2ee-0003.production-orch-1243.neo4j.io', 7687)))) => False
neo4j.pool                     DEBUG    [#0000]  _: <POOL> attempting to update routing table from IPv4Address(('p-ef39cb3b-a2ee-0003.prod

,event_number,event_type,agent_name
0,1,agent_reasoning,dummy-agent
1,2,tool_returned,dummy-agent


## Cleanup — delete this notebook's trace from Neo4j

In [11]:
neo4j.run_query(
    """
    MATCH (t:InvestigationTrace {trace_id: $trace_id})
    OPTIONAL MATCH (t)-[:HAS_EVENT]->(e:TraceEvent)
    DETACH DELETE t, e
    """,
    {"trace_id": trace.request_id},
)
print(f"Deleted trace {trace.request_id} and its events")

neo4j                          DEBUG    [#0000]  _: <WORKSPACE> routing towards fixed database: neo4j
neo4j                          DEBUG    [#0000]  _: <WORKSPACE> pinning database: 'neo4j'
neo4j.pool                     DEBUG    [#0000]  _: <POOL> acquire routing connection, access_mode='WRITE', database=AcquisitionDatabase(name='neo4j', guessed=False)
neo4j.pool                     DEBUG    [#0000]  _: <POOL> routing aged?, database=neo4j
neo4j.pool                     DEBUG    [#0000]  _: <ROUTING> purge check: last_updated_time=34757.973872208, ttl=10, perf_time=34775.624707125 => False
neo4j.pool                     DEBUG    [#0000]  _: <ROUTING> checking table freshness (readonly=False): table expired=True, has_server_for_mode=True, table routers=OrderedSet((IPv4Address(('p-ef39cb3b-a2ee-0003.production-orch-1243.neo4j.io', 7687)))) => False
neo4j.pool                     DEBUG    [#0000]  _: <POOL> attempting to update routing table from IPv4Address(('p-ef39cb3b-a2ee-0003.prod

Deleted trace 4cb03697-cf3c-4699-99ba-ba9ca5100e6d and its events


In [12]:
neo4j.close()
print("Driver closed")

neo4j.pool                     DEBUG    [#0000]  _: <POOL> close
neo4j.io                       DEBUG    [#C073]  C: GOODBYE
neo4j.io                       DEBUG    [#C073]  C: <CLOSE>
neo4j.io                       DEBUG    [#C081]  C: GOODBYE
neo4j.io                       DEBUG    [#C081]  C: <CLOSE>


Driver closed
